In [ ]:
import time
import subprocess
from pyspark.sql import SparkSession
from pyspark.sql.types import (
    StructType, StructField,
    TimestampType, IntegerType, StringType, DoubleType
)


In [ ]:
#HDFS paths
input_path = "hdfs://10.84.129.52:9000/aulas/fabricio_moreira/project/stream/input/"
checkpoint_path = "hdfs://10.84.129.52:9000/aulas/fabricio_moreira/project/stream/checkpoint/payment_format_counts"

In [ ]:
spark = SparkSession.builder \
    .appName("AML Big Data Processing") \
    .getOrCreate()

spark.sparkContext.setLogLevel("WARN")

In [ ]:
aml_schema = StructType([
    StructField("Timestamp", TimestampType(), True),
    StructField("From Bank", IntegerType(), True),
    StructField("Account2", StringType(), True),
    StructField("To Bank", IntegerType(), True),
    StructField("Account4", StringType(), True),
    StructField("Amount Received", DoubleType(), True),
    StructField("Receiving Currency", StringType(), True),
    StructField("Amount Paid", DoubleType(), True),
    StructField("Payment Currency", StringType(), True),
    StructField("Payment Format", StringType(), True),
    StructField("Is Laundering", IntegerType(), True)
])


In [ ]:
stream_df = spark.readStream \
    .option("header", True) \
    .option("timestampFormat", "yyyy/MM/dd HH:mm") \
    .schema(aml_schema) \
    .csv(input_path)

stream_df.printSchema()



In [ ]:
result = stream_df.groupBy("Payment Format").count()

In [ ]:
query = result.writeStream \
    .outputMode("complete") \
    .format("console") \
    .option("truncate", False) \
    .option("numRows", 50) \
    .option("checkpointLocation", checkpoint_path) \
    .start()

query.awaitTermination()

In [ ]:
query.stop()